In [46]:
import numpy as np

In [1]:
# load tahoe and sciplex3 deg outcomes

In [4]:
!ls ../../../theraformer/data/tahoe/dge/mDf2

A172_conditions.csv
A172_differential_expression_results.csv
A427_conditions.csv
A427_differential_expression_results.csv
A498_conditions.csv
A498_differential_expression_results.csv
A549_conditions.csv
A549_differential_expression_results.csv
AN3CA_conditions.csv
AN3CA_differential_expression_results.csv
AsPC1_conditions.csv
AsPC1_differential_expression_results.csv
BT474_conditions.csv
BT474_differential_expression_results.csv
C32_conditions.csv
C32_differential_expression_results.csv
C33A_conditions.csv
C33A_differential_expression_results.csv
CFPAC1_conditions.csv
CFPAC1_differential_expression_results.csv
CHP212_conditions.csv
CHP212_differential_expression_results.csv
COLO205_conditions.csv
COLO205_differential_expression_results.csv
H4_conditions.csv
H4_differential_expression_results.csv
HCT15_conditions.csv
HCT15_differential_expression_results.csv
HEC1A_conditions.csv
HEC1A_differential_expression_results.csv
HOP62_conditions.csv
HOP62_differential_expression_results.csv
HS57

In [114]:
import pandas as pd
cell_line_name = "A549"
deg_results = pd.read_csv(f"../../../theraformer/data/tahoe/dge/mDf2/{cell_line_name}_differential_expression_results.csv")
deg_results = deg_results[~deg_results['condition'].str.startswith('plate')]
deg_results['drug'] = deg_results['condition'] \
    .str.extract(r'^(.*?)_\d+_\d+_uM_')[0]

# 2) Extract “X_Y” as a single group, turn '_' → '.', then float
deg_results['dosage_uM'] = (
    deg_results['condition']
      .str.extract(r'_(\d+_\d+)_uM_')[0]         # gives e.g. "0_5"
      .str.replace('_', '.', regex=False)       # → "0.5"
      .astype(float)                             # → 0.5
)

In [115]:
deg_results

,logFC,AveExpr,t,P.Value,adj.P.Val,B,gene,condition,control_type,cellline,drug,dosage_uM
0,0.822508,6.912197,4.407047,1.692998e-05,0.048841,2.713301,TSC22D1,R_Verapamil_hydrochloride_0_05_uM_,"[('DMSO_TF', 0.0, 'uM')]",A549,R_Verapamil_hydrochloride,0.05
1,1.027384,4.891130,5.132570,6.645972e-07,0.017256,5.354536,NUMBL,R_Verapamil_hydrochloride_0_05_uM_,"[('DMSO_TF', 0.0, 'uM')]",A549,R_Verapamil_hydrochloride,0.05
2,1.515999,2.890899,4.683124,5.152209e-06,0.044591,3.004678,ABHD14B,R_Verapamil_hydrochloride_0_05_uM_,"[('DMSO_TF', 0.0, 'uM')]",A549,R_Verapamil_hydrochloride,0.05
3,0.973846,5.453270,4.481671,1.233928e-05,0.048841,2.899947,DNAJB1,R_Verapamil_hydrochloride_0_05_uM_,"[('DMSO_TF', 0.0, 'uM')]",A549,R_Verapamil_hydrochloride,0.05
4,-0.998162,5.560641,-4.434689,1.506512e-05,0.048841,2.637270,IER3IP1,R_Verapamil_hydrochloride_0_05_uM_,"[('DMSO_TF', 0.0, 'uM')]",A549,R_Verapamil_hydrochloride,0.05
...,...,...,...,...,...,...,...,...,...,...,...,...
333513,0.845514,6.878159,4.712448,4.526523e-06,0.020602,3.816013,PTPRK,XRK3F2_5_0_uM_,"[('DMSO_TF', 0.0, 'uM')]",A549,XRK3F2,5.00
333514,0.876376,6.149730,4.426020,1.562765e-05,0.045084,2.679927,RASEF,XRK3F2_5_0_uM_,"[('DMSO_TF', 0.0, 'uM')]",A549,XRK3F2,5.00
333515,3.084766,0.767461,6.071734,6.090928e-09,0.000158,-3.143918,ENSG00000176868,γ_Oryzanol_0_05_uM_,"[('DMSO_TF', 0.0, 'uM')]",A549,γ_Oryzanol,0.05
333516,-1.096922,5.102107,-5.197249,4.895521e-07,0.012711,-2.403103,ZNF621,γ_Oryzanol_0_5_uM_,"[('DMSO_TF', 0.0, 'uM')]",A549,γ_Oryzanol,0.50


In [79]:
import pandas as pd
import anndata

# add cell-line into condition
deg_results["condition"] = deg_results["condition"] + "_" + deg_results["cellline"]

# ──────────────────────────────────────────────────────
# 1) Pivot each metric so rows=condition, cols=gene
# ──────────────────────────────────────────────────────
metrics = {
    'logFC':     'logfc',
    'AveExpr':   'ave_expr',
    'P.Value':   'p_value',
    'adj.P.Val': 'adj_p_value',
    'B':         'b'
}

pivoted = {}
for orig_col, layer_name in metrics.items():
    pivoted[layer_name] = deg_results.pivot(
        index='condition',
        columns='gene',
        values=orig_col
    )

# capture the ordering
conditions = pivoted['logfc'].index
genes      = pivoted['logfc'].columns

# ──────────────────────────────────────────────────────
# 2) Build obs (one row per condition) and var (one row per gene)
# ──────────────────────────────────────────────────────
obs = (
    deg_results
    .drop_duplicates(subset=['condition'])
    .set_index('condition')
    [['control_type', 'cellline', 'drug', 'dosage_uM']]
    .loc[conditions]  # align order
)

# var with gene index and an explicit gene_name column
var = pd.DataFrame(index=genes)
var['gene_name'] = genes

# ──────────────────────────────────────────────────────
# 3) Create the AnnData, using logFC as X
# ──────────────────────────────────────────────────────
adata = anndata.AnnData(
    X   = pivoted['logfc'].loc[conditions, genes].values,
    obs = obs,
    var = var
)

# ──────────────────────────────────────────────────────
# 4) Stuff the other stats into layers
# ──────────────────────────────────────────────────────
for layer_name in ['ave_expr', 'p_value', 'adj_p_value', 'b']:
    adata.layers[layer_name] = (
        pivoted[layer_name]
        .loc[conditions, genes]
        .values
    )

adata.layers["logfc"] = adata.X
del adata.X

In [88]:
adata.write_h5ad("../../data/degs/tahoe_a549_only_sig.h5ad", compression="gzip")

In [91]:
tahoe_adata = adata

In [116]:
# load sciplex
import pandas as pd
cell_line_name = "A549"
deg_results = pd.read_csv(f"../../data/raw/sciplex/{cell_line_name}_differential_expression_results.csv")
deg_results = deg_results[~deg_results['condition'].str.startswith('plate')]
deg_results['drug'] = deg_results['condition'] \
    .str.extract(r'^(.*?)_\d+_\d+_uM_')[0]

# 2) Extract “X_Y” as a single group, turn '_' → '.', then float
deg_results['dosage_uM'] = (
    deg_results['condition']
      .str.extract(r'_(\d+_\d+)_uM_')[0]         # gives e.g. "0_5"
      .str.replace('_', '.', regex=False)       # → "0.5"
      .astype(float)                             # → 0.5
)
import anndata

# add cell-line into condition
deg_results["condition"] = deg_results["condition"] + "_" + deg_results["cellline"]

# ──────────────────────────────────────────────────────
# 1) Pivot each metric so rows=condition, cols=gene
# ──────────────────────────────────────────────────────
metrics = {
    'logFC':     'logfc',
    'AveExpr':   'ave_expr',
    'P.Value':   'p_value',
    'adj.P.Val': 'adj_p_value',
    'B':         'b'
}

pivoted = {}
for orig_col, layer_name in metrics.items():
    pivoted[layer_name] = deg_results.pivot(
        index='condition',
        columns='gene',
        values=orig_col
    )

# capture the ordering
conditions = pivoted['logfc'].index
genes      = pivoted['logfc'].columns

# ──────────────────────────────────────────────────────
# 2) Build obs (one row per condition) and var (one row per gene)
# ──────────────────────────────────────────────────────
obs = (
    deg_results
    .drop_duplicates(subset=['condition'])
    .set_index('condition')
    [['control_type', 'cellline', 'drug', 'dosage_uM']]
    .loc[conditions]  # align order
)

# var with gene index and an explicit gene_name column
var = pd.DataFrame(index=genes)
var['gene_name'] = genes

# ──────────────────────────────────────────────────────
# 3) Create the AnnData, using logFC as X
# ──────────────────────────────────────────────────────
adata = anndata.AnnData(
    X   = pivoted['logfc'].loc[conditions, genes].values,
    obs = obs,
    var = var
)

# ──────────────────────────────────────────────────────
# 4) Stuff the other stats into layers
# ──────────────────────────────────────────────────────
for layer_name in ['ave_expr', 'p_value', 'adj_p_value', 'b']:
    adata.layers[layer_name] = (
        pivoted[layer_name]
        .loc[conditions, genes]
        .values
    )

adata.layers["logfc"] = adata.X
del adata.X

/Users/arturszalata/miniconda3/envs/op3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [117]:
deg_results.head(5)

,logFC,AveExpr,t,P.Value,adj.P.Val,B,gene,condition,control_type,cellline,drug,dosage_uM
0,3.438906,1.493897,7.451840,1.788025e-13,2.288315e-09,14.548395,12756,factor_valid_drug_names_A_366_100_0_A549,factor_valid_drug_names_control_0_0,A549,NaN,NaN
1,2.251165,1.522012,4.387173,1.252221e-05,1.777579e-02,2.846893,4391,factor_valid_drug_names_A_366_1000_0_A549,factor_valid_drug_names_control_0_0,A549,NaN,NaN
2,3.732365,1.986303,5.311656,1.300274e-07,5.546969e-04,6.884712,4414,factor_valid_drug_names_A_366_1000_0_A549,factor_valid_drug_names_control_0_0,A549,NaN,NaN
3,2.619680,1.796392,4.080248,4.804811e-05,3.531789e-02,1.694732,6737,factor_valid_drug_names_A_366_1000_0_A549,factor_valid_drug_names_control_0_0,A549,NaN,NaN
4,3.290304,2.143300,4.079913,4.811660e-05,3.531789e-02,1.716146,9841,factor_valid_drug_names_A_366_1000_0_A549,factor_valid_drug_names_control_0_0,A549,NaN,NaN


In [97]:
from sklearn.metrics.pairwise import cosine_similarity

# 1) Get common genes between datasets
common_genes = np.intersect1d(adata.var_names, tahoe_adata.var_names)
adata_sub = adata[:, common_genes]
tahoe_sub = tahoe_adata[:, common_genes]

# 2) Get logfc and adj_p_value matrices
adata_lfc = adata_sub.layers["logfc"]
tahoe_lfc = tahoe_sub.layers["logfc"]
adata_padj = adata_sub.layers["adj_p_value"]
tahoe_padj = tahoe_sub.layers["adj_p_value"]

# Fill NaN values with 1 in adj_p_value matrices
adata_padj = np.nan_to_num(adata_padj, nan=1.0)
tahoe_padj = np.nan_to_num(tahoe_padj, nan=1.0)
adata_lfc = np.nan_to_num(adata_lfc, nan=0.0)
tahoe_lfc = np.nan_to_num(tahoe_lfc, nan=0.0)

# 3) Compute signed padj signatures
adata_signed_padj = np.sign(adata_lfc) * np.log10(adata_padj)
tahoe_signed_padj = np.sign(tahoe_lfc) * np.log10(tahoe_padj)

# 4) Calculate cosine similarity between all contrasts
cos_sim = cosine_similarity(adata_signed_padj, tahoe_signed_padj)

# 5) Create DataFrame with condition names
sim_df = pd.DataFrame(
    cos_sim,
    index=adata_sub.obs_names,
    columns=tahoe_sub.obs_names
)

sim_df

ValueError: Found array with 0 feature(s) (shape=(557, 0)) while a minimum of 1 is required by check_pairwise_arrays.

In [106]:
adata.var

,gene_name
gene,
1,1
2,2
3,3
4,4
5,5
...,...
12794,12794
12795,12795
12796,12796


In [99]:
tahoe_signed_padj

array([], shape=(923, 0), dtype=float64)